# 01 — Expression data (TCGA, recount3)

Downloads TCGA RNA-seq counts for one cancer from recount3 and turns them into the expression matrices used by every later notebook: counts → TPM → protein-coding genes → log2(TPM+1) → one row per gene name → one column per patient → z-score.
`{prefix}` = `CANCER.lower()` (e.g. `brca`). **Only the config lines in the first code cell need editing for a new cancer.**

## Inputs

Nothing local; everything is downloaded by the `recount3` package (needs internet, files are cached after the first run).

| Data | From | Format |
|---|---|---|
| Sample list | recount3, TCGA | one row per sample; `external_id`, `project`, ... |
| QC table (`recount_qc`) | recount3 | one row per sample; alignment stats, incl. `star.average_mapped_length` (average read length) |
| TCGA metadata table (`tcga`) | recount3 | one row per sample; `gdc_cases.samples.sample_type`, `tcga_barcode`, `gdc_cases.submitter_id`, ... |
| Gene counts | recount3 | genes (Ensembl IDs) × samples; coverage counts (bases summed over each gene), not read counts |
| Gene annotation (GTF) | recount3 | one row per gene; `gene_name`, `gene_type` (its `bp_length` is chromosome span and is not used) |
| Exon annotation (GTF) | recount3 | one row per exon; used to compute each gene's total exon length for TPM |

## Outputs (all in `base_dir`)

| File | Format |
|---|---|
| `{prefix}_tpm_unfiltered.csv` | genes (Ensembl IDs) × samples (recount3 `external_id`); TPM; all gene types, primary tumour samples only |
| `{prefix}_log2tpm.csv` | genes (symbols) × patients (`TCGA-XX-XXXX`); log2(TPM+1); protein-coding, duplicate gene names collapsed, one sample per patient, zero-variance genes removed; input to MINER in 02 |
| `{prefix}_zscored.csv` | same genes × patients; each gene z-scored across patients (mean 0, sd 1); used by 02 (patient list), 04, 05, 06 |


In [1]:
# Necessary imports:
import recount3
import pandas as pd
import numpy as np
import os

from recount3 import R3Resource, R3GeneOrExonCounts, R3Annotations
from recount3.bundle import _read_gtf_dataframe, _ranges_from_gtf
from collections import Counter
from scipy.sparse import csr_matrix
from scipy import stats
from scipy.sparse.csgraph import connected_components
 
# CANCER CONFIGURATION: the only lines to edit for a new cancer
# TCGA project code (BRCA, KIRC, LUAD, UCEC, THCA, PRAD, LUSC, HNSC, COAD, LGG)
CANCER   = "BRCA"

# folder holding this cancer's run; the three output files are written here
base_dir = r"causal_inf_new"

# this cancer's own folder: holds <prefix>_data, <prefix>_notebooks and <prefix>_output
cancer_dir = os.path.join(base_dir, CANCER.lower())
# this cancer's expression / mutation files (written by 01 and 02)
data_dir   = os.path.join(cancer_dir, f"{CANCER.lower()}_data")
prefix   = CANCER.lower()

os.makedirs(data_dir, exist_ok=True)
print(f"Cancer: {CANCER} | outputs: {data_dir}")

Cancer: BRCA | outputs: causal_inf_new


In [2]:
# Loading sample data for this cancer and printing metadata columns
samples = recount3.available_samples(organism="human", data_sources="tcga")

# Copy all records where a) organism is human b) data source is TCGA and c) the project is this cancer
samples_cancer = samples[samples["project"] == CANCER].copy()

# Overview of the data
print(f"Samples in {CANCER}: {len(samples_cancer)}")
print(f"\nAll metadata columns ({len(samples_cancer.columns)}):")

# List of columns
for c in samples_cancer.columns:
    print(f"  {c}")

# Should be ['external_id', 'project', 'organism', 'file_source', 'date_processed', 'project_home', 'project_type']

Samples in BRCA: 1256

All metadata columns (7):
  external_id
  project
  organism
  file_source
  date_processed
  project_home
  project_type


In [3]:
# Quality control (QC) and alignment stats are present in a diff metadata table than the one above
# Collecting these and then merging them into 1 single table

qc_resources = recount3.search_metadata_files(
    organism="human",
    data_source="tcga",
    table_name="recount_qc",
    project=CANCER,
)

# Sanity check, should return only 1 QC table per cancer
assert len(qc_resources) == 1, f"Expected exactly 1 QC metadata resource, got {len(qc_resources)}"

# Initialise this dataframe
qc_df = qc_resources[0].load()
print(f"QC table shape: {qc_df.shape}")

# Merge with the 1st table
samples_cancer = samples_cancer.merge(qc_df, on="external_id", how="left", suffixes=("", "_qc"))
print(f"After merging QC table: {samples_cancer.shape}")

# At this point the columns should be (have only shown first and last 5 cols):

# ['external_id', 'project', 'organism', 'file_source', 'date_processed', 
# ... (MANY COLUMNS) 'star.%_reads_mapped_to_too_many_loci_both', 'star.%_reads_unmapped:_other_both', 
# 'star.%_reads_unmapped:_too_many_mismatches_both', # 'star.%_reads_unmapped:_too_short_both', 'star.uniquely_mapped_reads_%_both']

QC table shape: (1256, 82)
After merging QC table: (1256, 88)


In [4]:
# Fetching a third metadata table with clinical data: tumour stage, sample type, etc.
tcga_resources = recount3.search_metadata_files(
    organism="human",
    data_source="tcga",
    table_name="tcga",
    project=CANCER,
)

# Sanity check
assert len(tcga_resources) == 1, f"Expected exactly 1 TCGA metadata resource, got {len(tcga_resources)}"
tcga_df = tcga_resources[0].load()
print(f"TCGA table shape: {tcga_df.shape}")

# Merge with the other 2 tables 
samples_cancer = samples_cancer.merge(tcga_df, on="external_id", how="left", suffixes=("", "_tcga"))
print(f"After merging TCGA table: {samples_cancer.shape}")

# Potential columns that might contain sample type 
candidate_cols = [c for c in samples_cancer.columns if "sample_type" in c.lower()]
for c in candidate_cols:
    print(f"\n  {c} — value counts:")
    print(samples_cancer[c].value_counts(dropna=False).to_string())

TCGA table shape: (1256, 840)
After merging TCGA table: (1256, 927)

  cgc_sample_sample_type — value counts:
cgc_sample_sample_type
Primary Tumor          1127
Solid Tissue Normal     112
NaN                      10
Metastatic                7

  gdc_cases.samples.sample_type_id — value counts:
gdc_cases.samples.sample_type_id
1.0     1127
11.0     112
NaN       10
6.0        7

  gdc_cases.samples.sample_type — value counts:
gdc_cases.samples.sample_type
Primary Tumor          1127
Solid Tissue Normal     114
Primary solid Tumor       8
Metastatic                7

  cgc_sample_sample_type_code — value counts:
cgc_sample_sample_type_code
1     1135
11     114
6        7


site-packages\recount3\resource.py:719: DtypeWarning: Columns (486,493,497,498,502) have mixed types. Specify dtype option on import or set low_memory=False.
  obj = pd.read_table(cached, compression="infer")


In [5]:
# Primary tumour filter:
primary_tumor_ids = samples_cancer.loc[
    samples_cancer["gdc_cases.samples.sample_type"] == "Primary Tumor", "external_id"]

print(f"Primary Tumor samples: {len(primary_tumor_ids)}")

# Mapped read: short RNA fragment whose position in genome has been found
# Read counts: no. of mapped reads
# Coverage counts: read counts * avg. mapped read length

# Average mapped read length per sample (from the STAR aligner QC table) is found below
# Needed later to turn recount3 coverage counts into read counts
AVG_READ_LEN_COL = "star.average_mapped_length"

# Safety check: if relevant column is missing, exit
if AVG_READ_LEN_COL not in samples_cancer.columns:
    
    raise SystemExit(
        f"STOP: expected column '{AVG_READ_LEN_COL}' not found in metadata. "
        f"Check the full column list printed above before proceeding."

    )

print(f"Found avg-read-length column: {AVG_READ_LEN_COL}")

# Set external_id to index and converts avg read counts to numeric. If the conversion doesn't work, forces a NaN
samples_cancer = samples_cancer.set_index("external_id")
avg_read_len = pd.to_numeric(samples_cancer[AVG_READ_LEN_COL], errors="coerce")
 
# Restrict to primary tumor samples only
avg_read_len = avg_read_len.loc[avg_read_len.index.intersection(primary_tumor_ids)]
print(f"avg_read_len restricted to primary tumor: {len(avg_read_len)}, "
      f"min={avg_read_len.min():.2f}, max={avg_read_len.max():.2f}, "
      f"missing={avg_read_len.isna().sum()}")

Primary Tumor samples: 1127
Found avg-read-length column: star.average_mapped_length
avg_read_len restricted to primary tumor: 1127, min=94.21, max=99.54, missing=0


In [6]:
# Raw gene-level counts for the project (genes x samples)
# recount3 stores coverage (bases summed over each gene), not read counts. These are found below:
count_resources = recount3.search_count_files_gene_or_exon(
    organism="human",
    data_source="tcga",
    genomic_unit="gene",
    project=CANCER,
)

# Should get exactly 1 coverage counts file
assert len(count_resources) == 1, f"Expected exactly 1 count resource, got {len(count_resources)}"
 
# Extracts the coverage counts file
count_res = count_resources[0]

# Gene annotation version used as reference while mapping counts
ann_ext = count_res.description.annotation_extension
print(f"Annotation extension used for counts: {ann_ext}")

# Loading coverage counts into a table; shape is (genes, samples)
raw_counts = count_res.load()
print(f"Raw counts shape: {raw_counts.shape}")
print(raw_counts.iloc[:3, :3])
 

Annotation extension used for counts: G026
Raw counts shape: (63856, 1256)
                   5b2072ee-b856-45c4-9c14-449654a40600  \
gene_id                                                   
ENSG00000278704.1                                     0   
ENSG00000277400.1                                     0   
ENSG00000274847.1                                     0   

                   2ba5b847-887f-415a-a7f3-94e6ea4d3800  \
gene_id                                                   
ENSG00000278704.1                                     0   
ENSG00000277400.1                                     0   
ENSG00000274847.1                                     0   

                   0b1cf6de-f98f-4020-baa7-c4fc518e5f00  
gene_id                                                  
ENSG00000278704.1                                     0  
ENSG00000277400.1                                     0  
ENSG00000274847.1                                     0  


In [7]:
# Gene annotation (GTF) matching the counts: gives each gene's length (bp_length), name and biotype
ann_resources = recount3.search_annotations(
    organism="human",
    genomic_unit="gene",
    annotation_extension=ann_ext,
)

assert len(ann_resources) == 1, f"Expected exactly 1 annotation resource, got {len(ann_resources)}"
 
ann_res = ann_resources[0]

# Cache the gene annotation locally
ann_res.download(path=None, cache_mode="enable") 
 
# Read the GTF and keep one row per gene, indexed by Ensembl gene ID
# Each row contains start, stop pos of the gene:

# Read gene annotation into table
gtf_df = _read_gtf_dataframe(ann_res)

# Keep only lines in the annotation describing whole genes
gene_ranges = _ranges_from_gtf(gtf_df, feature_kind="gene")

# Keeps only unique feature_ids (in this case, Ensembl IDs)
gene_ranges = gene_ranges.set_index("feature_id")

# Gene length for TPM = total EXON length (introns excluded), because RNA-seq reads only come from exons.
# The bp_length that _ranges_from_gtf returns is the gene's full span on the chromosome (introns included), which makes
# long genes look 50-100x less expressed than they are, so it is replaced here.
# Exon positions come from recount3's exon-level annotation of the same version as the counts.
exon_resources = recount3.search_annotations(
    organism="human",
    genomic_unit="exon",
    annotation_extension=ann_ext,
)
assert len(exon_resources) == 1, f"Expected exactly 1 exon annotation resource, got {len(exon_resources)}"
exon_res = exon_resources[0]
exon_res.download(path=None, cache_mode="enable")
exon_gtf = _read_gtf_dataframe(exon_res)

# One row per exon: gene ID, start, end
exons = exon_gtf[exon_gtf["feature"] == "exon"][["start", "end", "attributes"]].copy()
exons["gene_id"] = exons["attributes"].str.extract(r'gene_id "([^"]+)"')[0]
exons[["start", "end"]] = exons[["start", "end"]].astype(int)
exons = exons.sort_values(["gene_id", "start", "end"])

# Exons of different transcripts overlap, so each base is counted once:
# for every exon, add only the bases beyond the furthest end seen so far in that gene
furthest  = exons.groupby("gene_id")["end"].cummax()
prev_end  = furthest.groupby(exons["gene_id"]).shift().fillna(0)
new_bases = (exons["end"] - np.maximum(exons["start"] - 1, prev_end)).clip(lower=0)
exon_length = new_bases.groupby(exons["gene_id"]).sum()

# Keep the old value as span_length for reference and use exon length as bp_length from here on
gene_ranges["span_length"] = gene_ranges["bp_length"]
gene_ranges["bp_length"]   = exon_length.reindex(gene_ranges.index)
assert gene_ranges["bp_length"].notna().all(), "some genes have no exon length"
assert (gene_ranges["bp_length"] <= gene_ranges["span_length"]).all(), "exon length should never exceed the span"
del exon_gtf, exons


print(f"Genes with bp_length from GTF: {len(gene_ranges)}")
print(gene_ranges[["seqnames", "starts", "ends", "span_length", "bp_length"]].head())

Genes with bp_length from GTF: 63856
                     seqnames  starts    ends  span_length  bp_length
feature_id                                                           
ENSG00000278704.1  GL000009.2   56140   58376         2237     2237.0
ENSG00000277400.1  GL000194.1   53590  115018        61429     2179.0
ENSG00000274847.1  GL000194.1   53594  115055        61462     1599.0
ENSG00000277428.1  GL000195.1   37434   37534          101      101.0
ENSG00000276256.1  GL000195.1   42939   49164         6226     2195.0


In [8]:
# Align the three tables: keep genes that have a length, and samples that are primary tumour with a valid read length

# Genes for which there is data in BOTH gene annotation and coverage counts
common_genes = raw_counts.index.intersection(gene_ranges.index)

# Drops any samples with NaN coverage length. Also drops all tumours except primary
common_samples = raw_counts.columns.intersection(avg_read_len.dropna().index)

print(f"Genes in both counts and GTF: {len(common_genes)} / {raw_counts.shape[0]}")
print(f"Samples in both counts and metadata (primary tumor + valid read length): "
      f"{len(common_samples)} / {raw_counts.shape[1]}")
 
# Safety stop: if <95% of gene IDs match, the counts and the GTF use different ID formats
if len(common_genes) < 0.95 * raw_counts.shape[0]:
    print("\nExample raw_counts gene IDs:", raw_counts.index[:5].tolist())
    print("Example GTF gene IDs:", gene_ranges.index[:5].tolist())
    raise SystemExit(
        "STOP: fewer than 95% of gene IDs matched between counts and GTF. "
        "Compare the example IDs printed above before proceeding."
    )
 
# Same genes and samples, in the same order, in all three objects
raw_counts_aligned = raw_counts.loc[common_genes, common_samples]
bp_length = gene_ranges.loc[common_genes, "bp_length"].astype(float)
avg_read_len_aligned = avg_read_len.loc[common_samples]


Genes in both counts and GTF: 63856 / 63856
Samples in both counts and metadata (primary tumor + valid read length): 1127 / 1256


In [9]:
# TPM (transcripts per kilobase million) computation:

# Coverage / average read length = approximate read count
read_counts = raw_counts_aligned.div(avg_read_len_aligned, axis=1)

# Divide by gene length in kb to get reads per kilobase (RPK)
rpk = read_counts.div(bp_length / 1000.0, axis=0)

# Per-sample scaling factor = total RPM / 1 million
scale_factors = rpk.sum(axis=0) / 1_000_000.0 

# RPK / scaling factor = TPM (every sample sums to 1 million: sum * 1 million / sum)
# Finally tpm is df with rows = Ensembl IDs and columns = samples, labelled by sample ID
tpm = rpk.div(scale_factors, axis=1)                                
 
print(f"TPM matrix shape: {tpm.shape}")
print(tpm.iloc[:5, :5])

TPM matrix shape: (63856, 1127)
                    5b2072ee-b856-45c4-9c14-449654a40600  \
ENSG00000000003.14                             13.574156   
ENSG00000000005.5                               0.055572   
ENSG00000000419.12                             93.091082   
ENSG00000000457.13                             10.782028   
ENSG00000000460.16                              8.781498   

                    2ba5b847-887f-415a-a7f3-94e6ea4d3800  \
ENSG00000000003.14                             13.439751   
ENSG00000000005.5                               1.209980   
ENSG00000000419.12                             73.307216   
ENSG00000000457.13                             10.829303   
ENSG00000000460.16                              9.194091   

                    0b1cf6de-f98f-4020-baa7-c4fc518e5f00  \
ENSG00000000003.14                             36.651267   
ENSG00000000005.5                               0.127284   
ENSG00000000419.12                             83.809441   
ENSG00

In [10]:
# Save the unfiltered TPM matrix (all genes, primary tumour samples only)
 
tpm.to_csv(os.path.join(data_dir, f"{prefix}_tpm_unfiltered.csv"))
print(f"Saved unfiltered TPM matrix to {prefix}_tpm_unfiltered.csv")
print(f"Shape: {tpm.shape[0]} genes x {tpm.shape[1]} samples (primary tumor only)")

Saved unfiltered TPM matrix to brca_tpm_unfiltered.csv
Shape: 63856 genes x 1127 samples (primary tumor only)


In [11]:
# Check that the annotation has a gene_type (biotype) column, needed for the protein-coding filter
print("gene_ranges columns:", gene_ranges.columns.tolist())
if "gene_type" in gene_ranges.columns:
    print(gene_ranges["gene_type"].value_counts(dropna=False))
else:
    print("NO gene_type column — need a different source for biotype filtering.")

gene_ranges columns: ['seqnames', 'starts', 'ends', 'strand', 'source', 'type', 'bp_length', 'phase', 'gene_id', 'gene_name', 'gene_type', 'havana_gene', 'level', 'tag', 'span_length']
gene_type
protein_coding                        22321
processed_pseudogene                  10812
lincRNA                                7835
antisense                              5726
unprocessed_pseudogene                 3343
misc_RNA                               2391
snRNA                                  2053
miRNA                                  1892
TEC                                    1095
snoRNA                                 1006
sense_intronic                          932
transcribed_unprocessed_pseudogene      894
processed_transcript                    851
rRNA                                    569
transcribed_processed_pseudogene        511
IG_V_pseudogene                         290
IG_V_gene                               227
sense_overlapping                       192
TR_V_gene    

In [12]:
# Protein-coding genes only
protein_coding_ids = gene_ranges.loc[gene_ranges["gene_type"] == "protein_coding"].index

# Finds intersection of TPM table and protein coding gene table
# tpm_pc now has TPM of ONLY PROTEIN-CODING GENES
tpm_pc = tpm.loc[tpm.index.intersection(protein_coding_ids)]
print(f"After protein-coding filter: {tpm_pc.shape}")

# Conversion of TPM to log2 before deduplication. 1 is added because some genes have TPM = 0; log2(0) = -inf
log_tpm = np.log2(tpm_pc + 1)

# Map gene_id (Ensembl) to gene_name: extracts gene_name corresponding to log_tpm.index, which is a set of Ensembl IDs
gene_name_map = gene_ranges.loc[log_tpm.index, "gene_name"]
log_tpm_named = log_tpm.copy()

# Set the index to gene name, not ENsembl ID
log_tpm_named.index = gene_name_map.values
log_tpm_named.index.name = "gene_name"

# Several Ensembl IDs can share one gene symbol; these duplicates can now be collapsed
dup_genes = log_tpm_named.index[log_tpm_named.index.duplicated(keep=False)].unique()
print(f"Duplicate gene_name entries: {len(dup_genes)}")

# How many gene names have > 1 rows:

multiplicity = Counter(log_tpm_named.index.value_counts().loc[dup_genes])
print(f"Duplicate multiplicity breakdown (count of dup genes by how many rows they have): {dict(multiplicity)}")

After protein-coding filter: (22321, 1127)
Duplicate gene_name entries: 957
Duplicate multiplicity breakdown (count of dup genes by how many rows they have): {np.int64(2): 643, np.int64(10): 15, np.int64(8): 52, np.int64(3): 90, np.int64(6): 29, np.int64(7): 59, np.int64(5): 30, np.int64(4): 26, np.int64(33): 1, np.int64(31): 2, np.int64(12): 2, np.int64(35): 1, np.int64(16): 1, np.int64(9): 2, np.int64(36): 1, np.int64(42): 1, np.int64(13): 2}


In [13]:
# Collapse duplicate gene names into one row each

def collapse_group(sub_df, corr_threshold=0.8):

    '''

    Collapses the duplicate rows of one gene name into a single expression profile. Rows that are correlated with each other are 
    grouped into clusters and averaged; the cluster whose average varies most across samples is kept. Correlation is measured by
    Spearman coefficient

    Why not simple averaging? This is because the duplicates may happen to be different loci sharing the same symbol (unrelated)
    They may also be the same gene double-annotated or adjacent overlapping genes (related)

    Args: sub_df (df), shape (n_duplicates, n_samples). Rows are the Ensembl IDs sharing one gene name, contains log2(TPM+1)
          corr_threshold (float). Minimum correlation for two rows to be put in the same cluster (default 0.8)

    Returns: value (Series), shape (n_samples,). Collapsed expression of the gene
             n_clusters (int) number of clusters the duplicate rows split into
             largest_cluster_size (int) number of rows in the biggest cluster

    '''

    # Only one row: nothing to collapse
    n = sub_df.shape[0]
    if n == 1:
        return sub_df.iloc[0], 1, 1  # value, n_clusters, largest_cluster_size

    # Rows correlated >= threshold are linked; connected groups of linked rows form clusters
    # pairwise correlation across samples
    corr = sub_df.T.corr().values         
    adj = (corr >= corr_threshold).astype(int)
    np.fill_diagonal(adj, 1)
    n_components, labels = connected_components(csgraph=csr_matrix(adj), directed=False)

    # Average the rows of each cluster and record how much that average varies across samples
    cluster_means = []
    cluster_vars = []
    cluster_sizes = []
    for c in range(n_components):
        idx = np.where(labels == c)[0]
        cm = sub_df.iloc[idx].mean(axis=0)
        cluster_means.append(cm)
        cluster_vars.append(cm.var())
        cluster_sizes.append(len(idx))

    # Keep the cluster with the highest variance
    best = int(np.argmax(cluster_vars))
    return cluster_means[best], n_components, max(cluster_sizes)

# Genes with a unique name pass through unchanged
non_dup = log_tpm_named[~log_tpm_named.index.isin(dup_genes)]

# Collapse each duplicated gene name and keep diagnostics (number of clusters, largest cluster, group size)
collapsed_rows = []
diag_n_components = []
diag_largest_cluster = []
diag_group_size = []

for gene in dup_genes:
    sub = log_tpm_named.loc[[gene]] if not isinstance(log_tpm_named.loc[gene], pd.Series) else None
    sub = log_tpm_named.loc[gene]
    result, n_comp, largest = collapse_group(sub)
    result = result.copy()
    result.name = gene
    collapsed_rows.append(result)
    diag_n_components.append(n_comp)
    diag_largest_cluster.append(largest)
    diag_group_size.append(sub.shape[0])

collapsed_df = pd.DataFrame(collapsed_rows)
collapsed_df.index.name = "gene_name"

# Final matrix: unique genes + collapsed duplicates
log_tpm_clean = pd.concat([non_dup, collapsed_df])
print(f"Before collapsing: {log_tpm_named.shape}")
print(f"After collapsing: {log_tpm_clean.shape}")
print(f"Duplicate gene names remaining: {log_tpm_clean.index.duplicated().sum()}")

# Diagnostic table: how many duplicate groups were fully correlated vs split into several clusters
diag = pd.DataFrame({
    "gene_name": list(dup_genes),
    "group_size": diag_group_size,
    "n_clusters": diag_n_components,
    "largest_cluster": diag_largest_cluster,
})
print(f"\nGroups fully correlated (1 cluster): {(diag['n_clusters']==1).sum()} / {len(diag)}")
print(f"Groups that split into multiple clusters: {(diag['n_clusters']>1).sum()} / {len(diag)}")
print(diag.sort_values('group_size', ascending=False).head(10))

Before collapsing: (22321, 1127)
After collapsing: (20023, 1127)
Duplicate gene names remaining: 0

Groups fully correlated (1 cluster): 12 / 957
Groups that split into multiple clusters: 945 / 957
    gene_name  group_size  n_clusters  largest_cluster
821   KIR3DL3          42          42                1
816   KIR3DL2          36          36                1
562   KIR2DL4          35          35                1
165   KIR2DL1          33          33                1
364   KIR3DL1          31          31                1
825   KIR2DL3          31          31                1
565      NCR1          16          16                1
881   KIR2DL2          13          13                1
900   KIR2DS2          13          13                1
540      FCAR          12          12                1


In [14]:
# Check which metadata columns hold the TCGA barcode / patient ID
print([c for c in samples_cancer.columns if "barcode" in c.lower() or "submitter_id" in c.lower()])

print(samples_cancer[["tcga_barcode", "gdc_cases.submitter_id"]].head(10))

# Derive patient_id from tcga_barcode using the same convention already used elsewhere in this project
derived_patient_id = samples_cancer["tcga_barcode"].apply(lambda x: "-".join(str(x).split("-")[:3]))
print("\nDerived from tcga_barcode (first 3 fields):")
print(derived_patient_id.head(10))

print("\nDo derived_patient_id and gdc_cases.submitter_id match?")
print((derived_patient_id == samples_cancer["gdc_cases.submitter_id"]).value_counts(dropna=False))

['tcga_barcode', 'gdc_submitter_id', 'gdc_cases.submitter_id', 'gdc_cases.demographic.submitter_id', 'gdc_cases.diagnoses.submitter_id', 'gdc_cases.diagnoses.treatments.submitter_id', 'gdc_cases.samples.submitter_id', 'gdc_cases.samples.portions.submitter_id', 'gdc_cases.samples.portions.analytes.submitter_id', 'gdc_cases.samples.portions.analytes.aliquots.submitter_id', 'gdc_cases.samples.portions.annotations.submitter_id', 'gdc_cases.samples.annotations.submitter_id', 'gdc_cases.exposures.submitter_id', 'cgc_file_submitter_id', 'xml_bcr_patient_barcode']
                                                      tcga_barcode  \
external_id                                                          
0027045b-9ed6-45af-a68e-f55037b5184c  TCGA-B6-A0I8-01A-11R-A034-07   
0034ecfd-ca94-4615-94d2-e950a2b240ed  TCGA-AC-A2QI-01A-12R-A19W-07   
004e0c7a-a9f7-43ad-acac-52fb44f567fc  TCGA-A2-A0D2-01A-21R-A034-07   
00dd9349-a18d-4a08-8f90-639f64dc5ae6  TCGA-E9-A245-01A-22R-A16F-07   
00e9cce0-2367-496

In [15]:
# Rename sample columns to patient IDs and keep one sample per patient

# Samples where the two patient ID sources disagree (inspect with the commented print)
mismatch_mask = derived_patient_id != samples_cancer["gdc_cases.submitter_id"]
#print(samples_cancer.loc[mismatch_mask, ["tcga_barcode", "gdc_cases.submitter_id"]])

# Restrict to primary-tumor samples in the final deduped gene matrix
log_tpm_clean_pt = log_tpm_clean.loc[:, log_tpm_clean.columns.intersection(primary_tumor_ids)]
print(f"log_tpm_clean restricted to primary tumor: {log_tpm_clean_pt.shape}")

# Map external_id columns -> patient_id via tcga_barcode
barcode_map = samples_cancer.loc[log_tpm_clean_pt.columns, "tcga_barcode"]
patient_id_map = barcode_map.apply(lambda x: "-".join(str(x).split("-")[:3]))

# Rename columns from recount3 sample IDs to patient IDs (TCGA-XX-XXXX)
log_tpm_by_patient = log_tpm_clean_pt.copy()
log_tpm_by_patient.columns = patient_id_map.values

# Patients with more than one primary tumour sample
dupes = log_tpm_by_patient.columns[log_tpm_by_patient.columns.duplicated(keep=False)].unique().tolist()
print(f"Duplicate patients: {len(dupes)}")

# For each duplicated patient keep the sample with the highest median expression
if dupes:
    non_dup = log_tpm_by_patient.loc[:, ~log_tpm_by_patient.columns.isin(dupes)]
    best_samples = []
    for patient in dupes:
        cols = log_tpm_by_patient.loc[:, log_tpm_by_patient.columns == patient]
        best = cols.iloc[:, cols.median(axis=0).argmax()]
        best.name = patient
        best_samples.append(best)
    best_dup = pd.concat(best_samples, axis=1)
    log_tpm_final = pd.concat([non_dup, best_dup], axis=1)
else:
    log_tpm_final = log_tpm_by_patient

print(f"After patient dedup: {log_tpm_final.shape}")

log_tpm_clean restricted to primary tumor: (20023, 1127)
Duplicate patients: 12
After patient dedup: (20023, 1093)


In [16]:
# Drop zero-variance genes, z-score each gene across patients and save

# Genes with identical expression in every patient can't be z-scored (sd = 0)
zero_var = log_tpm_final.var(axis=1) == 0
print(f"Zero variance genes: {zero_var.sum()}")
log_tpm_final_nzv = log_tpm_final.loc[~zero_var]

# Z-score per gene (row): mean 0, sd 1 across patients
zscored = pd.DataFrame(
    stats.zscore(log_tpm_final_nzv.values, axis=1),
    index=log_tpm_final_nzv.index,
    columns=log_tpm_final_nzv.columns
)

print(f"\nFinal log2TPM shape: {log_tpm_final_nzv.shape}")
print(f"Final zscored shape: {zscored.shape}")
print(f"\nNaN in zscored: {zscored.isna().sum().sum()}")
print(f"Inf in zscored: {np.isinf(zscored.values).sum()}")

# Sanity check: per-gene mean should be ~0 and sd ~1
gene_means = zscored.mean(axis=1)
gene_stds = zscored.std(axis=1)
print(f"\nPer-gene mean - min:{gene_means.min():.4f} max:{gene_means.max():.4f} mean:{gene_means.mean():.4f}")
print(f"Per-gene std - min:{gene_stds.min():.4f} max:{gene_stds.max():.4f} mean:{gene_stds.mean():.4f}")

# Outputs: log2(TPM+1) is the input to MINER (notebook 02); the z-scored matrix (gene symbols x patients) is used by the later notebooks
log_tpm_final_nzv.to_csv(os.path.join(data_dir, f"{prefix}_log2tpm.csv"))
zscored.to_csv(os.path.join(data_dir, f"{prefix}_zscored.csv"))
print(f"\nSaved {prefix}_log2tpm.csv and {prefix}_zscored.csv")

Zero variance genes: 265

Final log2TPM shape: (19758, 1093)
Final zscored shape: (19758, 1093)

NaN in zscored: 0
Inf in zscored: 0

Per-gene mean - min:-0.0000 max:0.0000 mean:0.0000
Per-gene std - min:1.0005 max:1.0005 mean:1.0005

Saved brca_log2tpm.csv and brca_zscored.csv
